In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import os

In [17]:
class WDMClassifierLarge(nn.Module):
    """Large CNN for filament classification with deeper architecture"""
    def __init__(self, in_channels=1, num_classes=1, dropout=0.3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1),  
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1), 
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1), 
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=2, dilation=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=2, dilation=2),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # check this
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [18]:
class SimpleTransform:
    """Simple data transform for cosmic web images"""
    def __init__(self, size=(256, 256), normalize_stats=None, augment=False):
        self.size = size
        self.normalize_stats = normalize_stats
        self.augment = augment

    def __call__(self, img):
        # Convert to tensor if numpy
        if isinstance(img, np.ndarray):
            img = torch.from_numpy(img).float()
        
        # Add channel dimension if needed
        if img.dim() == 2:
            img = img.unsqueeze(0)
        
        # Resize
        img = F.interpolate(img.unsqueeze(0), size=self.size, mode='bilinear', align_corners=False)
        img = img.squeeze(0)
        
        # Apply log transform
        img = torch.clamp(img, min=0)
        img = torch.log1p(img)
        
        # Normalize
        if self.normalize_stats:
            img = (img - self.normalize_stats['mean']) / self.normalize_stats['std']
        
        # Simple augmentation for training
        if self.augment:
            if random.random() < 0.5:
                img = torch.flip(img, dims=[2])  # horizontal flip
            if random.random() < 0.5:
                img = torch.flip(img, dims=[1])  # vertical flip
            if random.random() < 0.5:
                k = random.choice([1, 2, 3])
                img = torch.rot90(img, k=k, dims=[1, 2])
        
        return img

In [19]:
class CosmicWebDataset(Dataset):
    """Dataset for cosmic web classification"""
    def __init__(self, cdm_data, wdm_data, indices, transform=None):
        self.cdm_data = cdm_data
        self.wdm_data = wdm_data
        self.indices = indices
        self.transform = transform
        
    def __len__(self):
        return len(self.indices) * 2  # CDM + WDM samples
    
    def __getitem__(self, idx):
        sample_idx = self.indices[idx // 2]
        is_wdm = idx % 2
        
        if is_wdm:
            image = self.wdm_data[sample_idx]
            label = 1.0
        else:
            image = self.cdm_data[sample_idx]
            label = 0.0
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.float32)

In [37]:
def evaluate_model(model, dataloader, criterion, device):
    """Evaluate model on given dataloader"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # FIX: Make consistent with training
            total_loss += loss.item() * labels.size(0)  # Changed from loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / total  # Changed from len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(cdm_file, wdm_file, config):
    """Main training function"""
    
    print("Loading data...")
    # Load and preprocess data
    cdm_data = np.load(cdm_file)
    wdm_data = np.load(wdm_file)
    
    # FIX: Split data FIRST, then calculate normalization stats
    n_samples = min(config['k_samples'], min(len(cdm_data), len(wdm_data)))
    indices = list(range(n_samples))
    random.shuffle(indices)
    
    train_end = int(0.6 * len(indices))
    val_end = int(0.8 * len(indices))
    
    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]
    
    print(f"Dataset split - Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")
    
    # FIX: Calculate normalization stats ONLY on training data
    train_cdm = cdm_data[train_indices]
    train_wdm = wdm_data[train_indices]
    train_data = np.concatenate((train_cdm, train_wdm))
    
    normalize_stats = {
        'mean': np.log1p(train_data).mean(),
        'std': np.log1p(train_data).std()
    }
    
    print(f"Data stats - Mean: {normalize_stats['mean']:.4f}, Std: {normalize_stats['std']:.4f}")
    
    # Create transforms
    train_transform = SimpleTransform(
        size=(config['img_size'], config['img_size']),
        normalize_stats=normalize_stats,
        augment=True
    )
    
    val_transform = SimpleTransform(
        size=(config['img_size'], config['img_size']),
        normalize_stats=normalize_stats,
        augment=False
    )
    
    # Create datasets
    train_dataset = CosmicWebDataset(cdm_data, wdm_data, train_indices, train_transform)
    val_dataset = CosmicWebDataset(cdm_data, wdm_data, val_indices, val_transform)
    test_dataset = CosmicWebDataset(cdm_data, wdm_data, test_indices, val_transform)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True,
        num_workers=2, 
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config['batch_size'], 
        shuffle=False,
        num_workers=2, 
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=config['batch_size'], 
        shuffle=False,
        num_workers=2, 
        pin_memory=True
    )
    
    # Setup model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = WDMClassifierLarge(dropout=config['dropout']).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Training loop
    train_losses = []
    val_losses = []
    val_accuracies = []
    best_val_acc = 0.0
    
    print(f"\nStarting training for {config['epochs']} epochs...")
    
    for epoch in range(config['epochs']):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
        for images, labels in progress_bar:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * labels.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
            # Update progress bar
            current_acc = train_correct / train_total
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_acc:.4f}'})
        
        # FIX: Calculate averages consistently
        avg_train_loss = train_loss / train_total  # Changed from len(train_dataset)
        train_acc = train_correct / train_total
        
        # Validation phase
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        
        # Store metrics
        train_losses.append(avg_train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        print(f"Epoch {epoch+1:3d}: Train Loss={avg_train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'config': config
            }, 'best_sample_model.pt')
            print(f"  -> New best model saved (Val Acc: {val_acc:.4f})")
    
    # Final evaluation
    print("\nLoading best model for final evaluation...")
    checkpoint = torch.load('best_sample_model.pt', map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_loss, test_acc = evaluate_model(model, test_loader, criterion, device)
    print(f"\nFinal Results:")
    print(f"Best Val Acc: {best_val_acc:.4f}")
    print(f"Test Acc: {test_acc:.4f}")

    return model, best_val_acc, test_acc

In [38]:
    # Configuration
    config = {
        'img_size': 256,
        'batch_size': 32,
        'lr': 2e-4,
        'weight_decay': 1e-4,
        'epochs': 20,
        'dropout': 0.1,
        'k_samples': 10000,  # Number of samples to use
    }
    
    cdm_file = '/n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy'
    wdm_file = '/n/netscratch/iaifi_lab/Lab/msliu/data/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy' # New WDM file
    # wdm_file = '/n/netscratch/iaifi_lab/Lab/ccuestalazaro/DREAMS/Images/WDM/boxes/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy'
    
    print("=== WDM Classification - Medium Model ===")
    print(f"Configuration: {config}")
    
    # Check if files exist
    if not os.path.exists(cdm_file):
        print(f"Error: CDM file not found: {cdm_file}")
        print("Please update the file paths in the script.")
        exit(1)
    
    if not os.path.exists(wdm_file):
        print(f"Error: WDM file not found: {wdm_file}")
        print("Please update the file paths in the script.")
        exit(1)
    
    # Train the model
    model, best_val_acc, test_acc = train_model(cdm_file, wdm_file, config)
    
    print(f"\nTraining completed!")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final test accuracy: {test_acc:.4f}")

=== WDM Classification - Medium Model ===
Configuration: {'img_size': 256, 'batch_size': 32, 'lr': 0.0002, 'weight_decay': 0.0001, 'epochs': 20, 'dropout': 0.1, 'k_samples': 10000}
Loading data...
Dataset split - Train: 6000, Val: 2000, Test: 2000
Data stats - Mean: 25.2856, Std: 1.1719
Using device: cuda
Model parameters: 4,503,681

Starting training for 20 epochs...


Epoch 1/20: 100%|████| 375/375 [01:41<00:00,  3.68it/s, loss=0.7056, acc=0.4974]


Epoch   1: Train Loss=0.6962, Train Acc=0.4974, Val Loss=0.6944, Val Acc=0.5000
  -> New best model saved (Val Acc: 0.5000)


Epoch 2/20: 100%|████| 375/375 [01:41<00:00,  3.68it/s, loss=0.6797, acc=0.5026]


Epoch   2: Train Loss=0.6943, Train Acc=0.5026, Val Loss=0.6932, Val Acc=0.5215
  -> New best model saved (Val Acc: 0.5215)


...

Epoch  18: Train Loss=0.0085, Train Acc=0.9987, Val Loss=0.0027, Val Acc=1.0000
  -> New best model saved (Val Acc: 1.0000)


Epoch 19/20: 100%|███| 375/375 [01:41<00:00,  3.69it/s, loss=0.0023, acc=0.9993]


Epoch  19: Train Loss=0.0058, Train Acc=0.9993, Val Loss=0.0038, Val Acc=0.9995


Epoch 20/20: 100%|███| 375/375 [01:41<00:00,  3.68it/s, loss=0.0028, acc=0.9992]


Epoch  20: Train Loss=0.0046, Train Acc=0.9992, Val Loss=0.0023, Val Acc=0.9998

Loading best model for final evaluation...

Final Results:
Best Val Acc: 1.0000
Test Acc: 0.9998

Training completed!
Best validation accuracy: 1.0000
Final test accuracy: 0.9998


In [39]:
    # Configuration
    config = {
        'img_size': 256,
        'batch_size': 32,
        'lr': 2e-4,
        'weight_decay': 1e-4,
        'epochs': 20,
        'dropout': 0.1,
        'k_samples': 10000,  # Number of samples to use
    }
    
    cdm_file = '/n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy'
    # wdm_file = '/n/netscratch/iaifi_lab/Lab/msliu/data/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy' # New WDM file
    wdm_file = '/n/netscratch/iaifi_lab/Lab/ccuestalazaro/DREAMS/Images/WDM/boxes/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy'
    
    print("=== WDM Classification - Medium Model ===")
    print(f"Configuration: {config}")
    
    # Check if files exist
    if not os.path.exists(cdm_file):
        print(f"Error: CDM file not found: {cdm_file}")
        print("Please update the file paths in the script.")
        exit(1)
    
    if not os.path.exists(wdm_file):
        print(f"Error: WDM file not found: {wdm_file}")
        print("Please update the file paths in the script.")
        exit(1)
    
    # Train the model
    model, best_val_acc, test_acc = train_model(cdm_file, wdm_file, config)
    
    print(f"\nTraining completed!")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final test accuracy: {test_acc:.4f}")

=== WDM Classification - Medium Model ===
Configuration: {'img_size': 256, 'batch_size': 32, 'lr': 0.0002, 'weight_decay': 0.0001, 'epochs': 20, 'dropout': 0.1, 'k_samples': 10000}
Loading data...
Dataset split - Train: 6000, Val: 2000, Test: 2000
Data stats - Mean: 25.2894, Std: 1.1721
Using device: cuda
Model parameters: 4,503,681

Starting training for 20 epochs...


Epoch 1/20: 100%|████| 375/375 [01:41<00:00,  3.68it/s, loss=0.6960, acc=0.5092]


Epoch   1: Train Loss=0.6948, Train Acc=0.5092, Val Loss=0.7009, Val Acc=0.5028
  -> New best model saved (Val Acc: 0.5028)


Epoch 2/20: 100%|████| 375/375 [01:41<00:00,  3.68it/s, loss=0.7605, acc=0.5282]


Epoch   2: Train Loss=0.6901, Train Acc=0.5282, Val Loss=0.7836, Val Acc=0.4935


...

Epoch  18: Train Loss=0.6449, Train Acc=0.6018, Val Loss=1.1095, Val Acc=0.4930


Epoch 19/20: 100%|███| 375/375 [01:41<00:00,  3.68it/s, loss=0.6303, acc=0.6040]


Epoch  19: Train Loss=0.6406, Train Acc=0.6040, Val Loss=0.8085, Val Acc=0.5282


Epoch 20/20: 100%|███| 375/375 [01:41<00:00,  3.68it/s, loss=0.6518, acc=0.6076]


Epoch  20: Train Loss=0.6383, Train Acc=0.6076, Val Loss=1.4333, Val Acc=0.5005

Loading best model for final evaluation...

Final Results:
Best Val Acc: 0.5740
Test Acc: 0.5753

Training completed!
Best validation accuracy: 0.5740
Final test accuracy: 0.5753
